In [16]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.inception_v3 import preprocess_input
from nltk.translate.bleu_score import corpus_bleu
import os
import string
import matplotlib.pyplot as plt
from PIL import Image


In [17]:
def extract_features(directory):
    model = InceptionV3(weights='imagenet')
    model = Model(inputs=model.input, outputs=model.layers[-2].output)
    features = {}
    valid_extensions = ('.jpg', '.jpeg', '.png')  # Add other valid extensions if needed

    for img_name in os.listdir(directory):
        if img_name.lower().endswith(valid_extensions):
            img_path = os.path.join(directory, img_name)
            try:
                img = image.load_img(img_path, target_size=(299, 299))
                img = image.img_to_array(img)
                img = np.expand_dims(img, axis=0)
                img = preprocess_input(img)
                feature = model.predict(img, verbose=0)
                img_id = img_name.split('.')[0]
                features[img_id] = feature
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
    return features

# Replace with the correct path to your image directory
image_directory = '/kaggle/input/flickr-images/flickr30k-images'
features = extract_features(image_directory)
print('Extracted Features: %d' % len(features))


96112376/96112376 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


I0000 00:00:1719245530.949195     104 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Extracted Features: 31783


In [52]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add, AdditiveAttention, RepeatVector
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.inception_v3 import preprocess_input
from nltk.translate.bleu_score import corpus_bleu
import os
import string
import matplotlib.pyplot as plt
from PIL import Image
import random
import keras_tuner as kt

def load_doc(filename):
    with open(filename, 'r') as file:
        text = file.read()
    return text

def load_descriptions(doc):
    mapping = {}
    for line in doc.split('\n'):
        tokens = line.split('\t')
        if len(tokens) < 2:
            continue
        img_id, caption = tokens[0], tokens[1]
        img_id = img_id.split('.')[0]
        if img_id not in mapping:
            mapping[img_id] = []
        mapping[img_id].append(caption)
    return mapping

def clean_descriptions(descriptions):
    table = str.maketrans('', '', string.punctuation)
    for key, desc_list in descriptions.items():
        for i in range(len(desc_list)):
            desc = desc_list[i]
            desc = desc.split()
            desc = [word.lower() for word in desc]
            desc = [w.translate(table) for w in desc]
            desc = [word for word in desc if len(word) > 1]
            desc = [word for word in desc if word.isalpha()]
            desc_list[i] = ' '.join(desc)

def to_vocabulary(descriptions):
    all_desc = set()
    for key in descriptions.keys():
        [all_desc.update(d.split()) for d in descriptions[key]]
    return all_desc

def save_descriptions(descriptions, filename):
    lines = list()
    for key, desc_list in descriptions.items():
        for desc in desc_list:
            lines.append(key + '\t' + desc)
    data = '\n'.join(lines)
    with open(filename, 'w') as file:
        file.write(data)

# Load descriptions
filename = '/kaggle/input/flickrtokens/results_20130124.token'
doc = load_doc(filename)
descriptions = load_descriptions(doc)
print('Loaded: %d ' % len(descriptions))
clean_descriptions(descriptions)
vocabulary = to_vocabulary(descriptions)
print('Vocabulary Size: %d' % len(vocabulary))
save_descriptions(descriptions, 'descriptions.txt')


def max_length(descriptions):
    lines = [d for desc in descriptions.values() for d in desc]
    return max(len(d.split()) for d in lines)

max_length = max_length(descriptions)
print('Description Length: %d' % max_length)

Loaded: 31783 
Vocabulary Size: 19735
Description Length: 72


In [53]:
def create_tokenizer(descriptions, vocab_size):
    lines = [d for desc in descriptions.values() for d in desc]
    tokenizer = Tokenizer(num_words=vocab_size)
    tokenizer.fit_on_texts(lines)
    return tokenizer

# Example usage
tokenizer = create_tokenizer(train_descriptions, vocab_size)
vocab_size = len(tokenizer.word_index) + 1
print(f"Tokenizer created with vocab_size: {vocab_size}")


Tokenizer created with vocab_size: 17025


In [55]:
# Split descriptions into training, validation, and test sets
all_img_ids = list(descriptions.keys())
random.shuffle(all_img_ids)

train_split = int(len(all_img_ids) * 0.7)
val_split = int(len(all_img_ids) * 0.9)

train_ids = all_img_ids[:train_split]
val_ids = all_img_ids[train_split:val_split]
test_ids = all_img_ids[val_split:]

train_descriptions = {k: descriptions[k] for k in train_ids}
val_descriptions = {k: descriptions[k] for k in val_ids}
test_descriptions = {k: descriptions[k] for k in test_ids}

train_features = {k: features[k] for k in train_ids}
val_features = {k: features[k] for k in val_ids}
test_features = {k: features[k] for k in test_ids}

print(f"Training set size: {len(train_descriptions)}")
print(f"Validation set size: {len(val_descriptions)}")
print(f"Test set size: {len(test_descriptions)}")

Training set size: 22248
Validation set size: 6356
Test set size: 3179


In [64]:
def create_sequences(tokenizer, max_length, desc_list, photo, vocab_size):
    X1, X2, y = [], [], []
    for desc in desc_list:
        seq = tokenizer.texts_to_sequences([desc])[0]
        for i in range(1, len(seq)):
            in_seq, out_seq = seq[:i], seq[i]
            in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
            out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
            X1.append(photo)
            X2.append(in_seq)
            y.append(out_seq)
    return np.array(X1), np.array(X2), np.array(y)

def data_generator(descriptions, photos, tokenizer, max_length, vocab_size):
    while True:
        for key, desc_list in descriptions.items():
            photo = photos[key][0]
            in_img, in_seq, out_word = create_sequences(tokenizer, max_length, desc_list, photo, vocab_size)
            for i in range(len(in_img)):
                yield (in_img[i], in_seq[i]), out_word[i]


In [65]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Embedding, Dropout, add
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.inception_v3 import preprocess_input
from nltk.translate.bleu_score import corpus_bleu
import os
import string
import matplotlib.pyplot as plt
from PIL import Image

In [66]:
def define_lstm_model(vocab_size, max_length, embedding_dim=256, units=256):
    inputs1 = Input(shape=(2048,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(units, activation='relu')(fe1)
    
    inputs2 = Input(shape=(max_length,))
    se1 = Embedding(vocab_size, embedding_dim, mask_zero=True)(inputs2)
    se2 = Dropout(0.5)(se1)
    se3 = LSTM(units, return_sequences=False, use_cudnn=False)(se2)
    
    decoder1 = add([fe2, se3])
    decoder2 = Dense(units, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)
    
    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model


In [67]:
def define_transformer_model(vocab_size, max_length, embedding_dim=256, units=256):
    inputs1 = Input(shape=(2048,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(units, activation='relu')(fe1)
    
    inputs2 = Input(shape=(max_length,))
    se1 = Embedding(vocab_size, embedding_dim, mask_zero=True)(inputs2)
    se2 = Dropout(0.5)(se1)
    
    mha = MultiHeadAttention(num_heads=8, key_dim=embedding_dim)(se2, se2, se2)
    mha = LayerNormalization(epsilon=1e-6)(mha + se2)
    
    mean_mha = ReduceMeanLayer(axis=1)(mha)
    
    decoder1 = add([fe2, mean_mha])
    decoder2 = Dense(units, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)
    
    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model


In [68]:
def define_gru_model(vocab_size, max_length, embedding_dim=256, units=256):
    inputs1 = Input(shape=(2048,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(units, activation='relu')(fe1)
    
    inputs2 = Input(shape=(max_length,))
    se1 = Embedding(vocab_size, embedding_dim, mask_zero=True)(inputs2)
    se2 = Dropout(0.5)(se1)
    se3 = GRU(units, return_sequences=False, use_cudnn=False)(se2)
    
    decoder1 = add([fe2, se3])
    decoder2 = Dense(units, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)
    
    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model


In [69]:
def get_output_signature(vocab_size):
    return (
        (tf.TensorSpec(shape=(2048,), dtype=tf.float32), tf.TensorSpec(shape=(None,), dtype=tf.int32)),
        tf.TensorSpec(shape=(vocab_size,), dtype=tf.float32)
    )


In [72]:
from sklearn.model_selection import ParameterGrid

# Define hyperparameters
param_grid = {
    'vocab_size': [5000, 10000, 20000],  # Ensure vocab_size matches the dataset's actual vocabulary size
    'max_length': [34, 72],  # Use the correct values for max_length
    'embedding_dim': [256, 512],
    'units': [256, 512]
}

# Define a function to train and evaluate models
def train_and_evaluate(model_fn, params):
    tokenizer = create_tokenizer(train_descriptions, params['vocab_size'])
    vocab_size = len(tokenizer.word_index) + 1
    print(f"Training with vocab_size: {vocab_size}")
    
    model = model_fn(vocab_size, params['max_length'], params['embedding_dim'], params['units'])
    
    output_signature = get_output_signature(vocab_size)
    
    train_dataset = tf.data.Dataset.from_generator(
        lambda: data_generator(train_descriptions, train_features, tokenizer, params['max_length'], vocab_size),
        output_signature=output_signature
    ).batch(32).repeat()
    
    val_dataset = tf.data.Dataset.from_generator(
        lambda: data_generator(val_descriptions, val_features, tokenizer, params['max_length'], vocab_size),
        output_signature=output_signature
    ).batch(32).repeat()
    
    steps_per_epoch = len(train_descriptions) // 32
    validation_steps = len(val_descriptions) // 32
    
    for epoch in range(10):
        print(f"Training epoch {epoch+1}/10 for model {model_fn.__name__} with parameters: {params}")
        model.fit(train_dataset, epochs=1, steps_per_epoch=steps_per_epoch, validation_data=val_dataset, validation_steps=validation_steps, verbose=2)
    
    score = model.evaluate(val_dataset, steps=validation_steps, verbose=0)
    if isinstance(score, list):
        score = score[0]  # Use the loss value if score is a list
    return score

# Hyperparameter tuning for all models
best_score = float('inf')
best_model_fn = None
best_params = None

for params in ParameterGrid(param_grid):
    for model_fn in [define_lstm_model, define_gru_model, define_transformer_model]:
        try:
            score = train_and_evaluate(model_fn, params)
            if score < best_score:  # Compare the loss value
                best_score = score
                best_model_fn = model_fn
                best_params = params
        except ValueError as e:
            print(f"Error with params: {params}, {str(e)}")

print(f"Best model function: {best_model_fn.__name__} with params: {best_params} and score: {best_score}")


Training with vocab_size: 17041
Training epoch 1/10 for model define_lstm_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}


W0000 00:00:1719251093.004745     105 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update
W0000 00:00:1719251093.011155     105 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update
W0000 00:00:1719251104.718898     106 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


695/695 - 20s - 29ms/step - loss: 6.6208 - val_loss: 5.9324
Training epoch 2/10 for model define_lstm_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/695 - 14s - 20ms/step - loss: 5.4552 - val_loss: 5.7111
Training epoch 3/10 for model define_lstm_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/695 - 14s - 20ms/step - loss: 4.9187 - val_loss: 5.6972
Training epoch 4/10 for model define_lstm_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/695 - 14s - 20ms/step - loss: 4.5621 - val_loss: 5.6786
Training epoch 5/10 for model define_lstm_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/695 - 14s - 20ms/step - loss: 4.3164 - val_loss: 5.8630
Training epoch 6/10 for model define_lstm_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}


W0000 00:00:1719251246.386639     105 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update
W0000 00:00:1719251246.391795     105 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update
W0000 00:00:1719251258.165687     105 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


695/695 - 21s - 30ms/step - loss: 6.5871 - val_loss: 5.8736
Training epoch 2/10 for model define_gru_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/695 - 14s - 20ms/step - loss: 5.3439 - val_loss: 5.6326
Training epoch 3/10 for model define_gru_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/695 - 14s - 20ms/step - loss: 4.8025 - val_loss: 5.7123
Training epoch 4/10 for model define_gru_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/695 - 14s - 20ms/step - loss: 4.4412 - val_loss: 5.8618
Training epoch 5/10 for model define_gru_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/695 - 14s - 20ms/step - loss: 4.1663 - val_loss: 6.1474
Training epoch 6/10 for model define_gru_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/6

...

695/695 - 46s - 66ms/step - loss: 6.6973 - val_loss: 5.9656
Training epoch 2/10 for model define_lstm_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}
695/695 - 39s - 56ms/step - loss: 5.3701 - val_loss: 5.7235
Training epoch 3/10 for model define_lstm_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}
695/695 - 39s - 56ms/step - loss: 4.8014 - val_loss: 5.9340
Training epoch 4/10 for model define_lstm_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}
695/695 - 39s - 56ms/step - loss: 4.3714 - val_loss: 6.1816
Training epoch 5/10 for model define_lstm_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}
695/695 - 39s - 56ms/step - loss: 4.0288 - val_loss: 6.5370
Training epoch 6/10 for model define_lstm_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20

W0000 00:00:1719263813.657911     106 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


695/695 - 37s - 53ms/step - loss: 6.6066 - val_loss: 5.8450
Training epoch 2/10 for model define_gru_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}
695/695 - 30s - 44ms/step - loss: 5.1956 - val_loss: 5.6572
Training epoch 3/10 for model define_gru_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}
695/695 - 30s - 44ms/step - loss: 4.5382 - val_loss: 6.0081
Training epoch 4/10 for model define_gru_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}
695/695 - 30s - 44ms/step - loss: 4.0715 - val_loss: 6.4059
Training epoch 5/10 for model define_gru_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}
695/695 - 30s - 44ms/step - loss: 3.7350 - val_loss: 6.8681
Training epoch 6/10 for model define_gru_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}


W0000 00:00:1719264128.953277     106 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update
W0000 00:00:1719264163.404353     105 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


695/695 - 44s - 63ms/step - accuracy: 0.0522 - loss: 6.8227 - val_accuracy: 0.0548 - val_loss: 6.4160
Training epoch 2/10 for model define_transformer_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}
695/695 - 36s - 52ms/step - accuracy: 0.0523 - loss: 6.1755 - val_accuracy: 0.0548 - val_loss: 6.5118
Training epoch 3/10 for model define_transformer_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}
695/695 - 36s - 51ms/step - accuracy: 0.0522 - loss: 6.0952 - val_accuracy: 0.0548 - val_loss: 6.5593
Training epoch 4/10 for model define_transformer_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}
695/695 - 36s - 51ms/step - accuracy: 0.0534 - loss: 5.9829 - val_accuracy: 0.0548 - val_loss: 6.6696
Training epoch 5/10 for model define_transformer_model with parameters: {'embedding_dim': 512, 'max_length': 72, 'units': 512, 'vocab_size': 20000}


In [74]:
tokenizer = create_tokenizer(train_descriptions, best_params['vocab_size'])
vocab_size = len(tokenizer.word_index) + 1

best_model = best_model_fn(vocab_size, best_params['max_length'], best_params['embedding_dim'], best_params['units'])

output_signature = get_output_signature(vocab_size)

train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(train_descriptions, train_features, tokenizer, best_params['max_length'], vocab_size),
    output_signature=output_signature
).batch(32).repeat()

val_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(val_descriptions, val_features, tokenizer, best_params['max_length'], vocab_size),
    output_signature=output_signature
).batch(32).repeat()

epochs = 20
steps_per_epoch = len(train_descriptions) // 32
validation_steps = len(val_descriptions) // 32

for epoch in range(epochs):
    print(f"Training epoch {epoch+1}/{epochs} for best model {best_model_fn.__name__} with parameters: {best_params}")
    best_model.fit(train_dataset, epochs=1, steps_per_epoch=steps_per_epoch, validation_data=val_dataset, validation_steps=validation_steps, verbose=1)

Training epoch 1/20 for best model define_transformer_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
 10/695 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.0106 - loss: 9.2361  

W0000 00:00:1719265550.979757     106 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


693/695 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0449 - loss: 7.0155

W0000 00:00:1719265563.651652     106 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


695/695 ━━━━━━━━━━━━━━━━━━━━ 22s 23ms/step - accuracy: 0.0450 - loss: 7.0139 - val_accuracy: 0.0570 - val_loss: 6.2637
Training epoch 2/20 for best model define_transformer_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/695 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.0533 - loss: 5.8814 - val_accuracy: 0.0623 - val_loss: 6.2840
Training epoch 3/20 for best model define_transformer_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/695 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.0656 - loss: 5.7488 - val_accuracy: 0.0683 - val_loss: 6.3429
Training epoch 4/20 for best model define_transformer_model with parameters: {'embedding_dim': 256, 'max_length': 34, 'units': 256, 'vocab_size': 5000}
695/695 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - accuracy: 0.0746 - loss: 5.7044 - val_accuracy: 0.0707 - val_loss: 6.3421
Training epoch 5/20 for best model define_transformer_model with par

In [88]:
def beam_search_predictions(model, tokenizer, photo, max_length, beam_index=3):
    start = [tokenizer.word_index['startseq']]
    start_word = [[start, 0.0]]
    
    while len(start_word[0][0]) < max_length:
        temp = []
        for s in start_word:
            par_caps = pad_sequences([s[0]], maxlen=max_length, padding='post')
            e = model.predict([photo, par_caps], verbose=0)
            word_preds = np.argsort(e[0])[-beam_index:]
            
            for w in word_preds:
                next_cap, prob = s[0][:], s[1]
                next_cap.append(w)
                prob += e[0][w]
                temp.append([next_cap, prob])
        
        start_word = temp
        start_word = sorted(start_word, reverse=False, key=lambda l: l[1])
        start_word = start_word[-beam_index:]
    
    start_word = start_word[-1][0]
    intermediate_caption = [tokenizer.index_word[i] for i in start_word]
    
    final_caption = []
    for word in intermediate_caption:
        if word != 'endseq':
            final_caption.append(word)
        else:
            break
    
    return ' '.join(final_caption[1:])


In [89]:
def generate_caption(model, tokenizer, photo, max_length):
    in_text = 'startseq'
    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)
        yhat = model.predict([photo, sequence], verbose=0)
        yhat = np.argmax(yhat)
        word = tokenizer.index_word.get(yhat, None)
        if word is None:
            break
        in_text += ' ' + word
        if word == 'endseq':
            break
    final_caption = in_text.split()
    final_caption = final_caption[1:-1]  # Remove 'startseq' and 'endseq'
    return ' '.join(final_caption)

def evaluate_model_batch(model, descriptions, photos, tokenizer, max_length, batch_size=32):
    actual, predicted = [], []
    keys = list(descriptions.keys())
    for i in range(0, len(keys), batch_size):
        batch_keys = keys[i:i+batch_size]
        batch_photos = [photos[key][0] for key in batch_keys]
        batch_descriptions = [descriptions[key] for key in batch_keys]
        yhat_texts = generate_desc_batch(model, tokenizer, batch_photos, max_length)
        for j, key in enumerate(batch_keys):
            yhat = yhat_texts[j]
            references = [d.split() for d in batch_descriptions[j]]
            actual.append(references)
            predicted.append(yhat.split())
        print(f"Processed {i}/{len(descriptions)} images")
    bleu1 = corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0))
    bleu2 = corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0))
    bleu3 = corpus_bleu(actual, predicted, weights=(0.33, 0.33, 0.33, 0))
    bleu4 = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25))
    return bleu1, bleu2, bleu3, bleu4

# Evaluate the best model
bleu1, bleu2, bleu3, bleu4 = evaluate_model_batch(best_model, val_descriptions, val_features, tokenizer, best_params['max_length'])
print(f'Best model Validation BLEU-1: {bleu1}')
print(f'Best model Validation BLEU-2: {bleu2}')
print(f'Best model Validation BLEU-3: {bleu3}')
print(f'Best model Validation BLEU-4: {bleu4}')


Processed 0/6356 images
Processed 32/6356 images
Processed 64/6356 images
Processed 96/6356 images
Processed 128/6356 images
Processed 160/6356 images
Processed 192/6356 images
Processed 224/6356 images
Processed 256/6356 images
Processed 288/6356 images
Processed 320/6356 images
Processed 352/6356 images
Processed 384/6356 images
Processed 416/6356 images
Processed 448/6356 images
Processed 480/6356 images
Processed 512/6356 images
Processed 544/6356 images
Processed 576/6356 images
Processed 608/6356 images
Processed 640/6356 images
Processed 672/6356 images
Processed 704/6356 images
Processed 736/6356 images
Processed 768/6356 images
Processed 800/6356 images
Processed 832/6356 images
Processed 864/6356 images
Processed 896/6356 images
Processed 928/6356 images
Processed 960/6356 images
Processed 992/6356 images
Processed 1024/6356 images
Processed 1056/6356 images
Processed 1088/6356 images
Processed 1120/6356 images
Processed 1152/6356 images
Processed 1184/6356 images
Processed 1